# Microdata choice training: datasets and run inventory

Summary of training, validation, and held-out test datasets (`Table 0a`) and canonical run/`test_best` completion status (`Table 0b`).

In [1]:
import sys
from pathlib import Path
from IPython.display import display

PAPER_DIR = next(
    (
        p
        for p in (
            Path.cwd(),
            *Path.cwd().parents,
            Path.cwd() / "train-notebooks" / "beta-training" / "ms_paper_beta_loss",
            Path.cwd() / "code" / "train-notebooks" / "beta-training" / "ms_paper_beta_loss",
        )
        if (p / "microdata_analysis.py").is_file()
    ),
    None,
)
if PAPER_DIR is None:
    raise FileNotFoundError(f"Could not locate microdata_analysis.py from {Path.cwd()}")
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

import microdata_analysis as ma

ctx = ma.load_microdata_context()
print(f"Loaded {len(ctx.regimes_by_model)} models from {ma.PAPER_RUNS_ROOT}")


Loaded 3 models from /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/runs-rorqual


In [2]:
import pandas as pd
from beta_ols import berke2023 as ds_berke
from beta_ols import covid as ds_covid
from beta_ols import grummon2025 as ds_grummon
from beta_ols import kantorowicz2026 as ds_kantorowicz
from beta_ols import kriner2026 as ds_kriner
from beta_ols import magistro2025 as ds_magistro
from beta_ols import wu2025 as ds_wu

DATASET_SPECS = (
    ("Electricity", "Wu (2025)", "Energy Research & Social Science", "Training (microdata)", "Electricity-plan conjoint: monthly bill, renewable share, local renewables", ds_wu, "CaseID", False),
    ("Drones", "Berke et al. (2023)", "Transportation Research Part C", "Training (microdata)", "Drone-delivery discrete choice: delivery fee, wait time, privacy", ds_berke, "ResponseId", False),
    ("Flights", "Goldfarb et al. (2026)", "Journal of Cleaner Production", "Validation (selects best.pt)", "Flight conjoint: price, sustainable aviation fuel, airline, baggage, legroom, seat", ds_kriner, "respondent", True),
    ("Consumer rights", "Kantorowicz-Reznichenko et al. (2026)", "Journal of Law and Empirical Analysis", "Held-out test (price study)", "Laptop & laundry-machine conjoints: price, brand, product specs, warranty, privacy, liability, conflict resolution", ds_kantorowicz, "CASEID", True),
    ("AI & offshoring", "Magistro et al. (2025)", "American Journal of Political Science", "Held-out test (price study)", "Smartphone AI & offshoring conjoint (US & Canada): price, customer service jobs, factory jobs, data science jobs", ds_magistro, "CASEID", True),
    ("Food labels", "Grummon et al. (2025)", "JAMA Network Open", "Held-out test (non-price study)", "Six-arm nutrition-label trial: front-of-package warning labels vs. control", ds_grummon, "participantid", True),
    ("Vaccines", "Kreps et al. (2020)", "JAMA Network Open", "Held-out test (non-price study)", "COVID-19 vaccine conjoint: efficacy, duration, side effects, FDA, origin, endorsements", ds_covid, "respondent", True),
)

dataset_rows = []
for shorthand, citation, journal, role, task_desc, mod, resp_col, uses_slopes in DATASET_SPECS:
    choice_df = mod.load_choice_df()
    n_resp = int(choice_df[resp_col].nunique())
    n_choices = int(len(choice_df))
    if uses_slopes:
        specs = mod.regression_specs(mod.regression_rows(choice_df))
        slopes_str = str(sum(len(spec.target_labels) for spec in specs.values()))
    else:
        slopes_str = "—"
    dataset_rows.append(
        {
            "Dataset": shorthand,
            "Citation": citation,
            "Journal": journal,
            "Role": role,
            "Choice task & key attributes": task_desc,
            "Respondents": f"{n_resp:,}",
            "Choices": f"{n_choices:,}",
            "OLS slopes": slopes_str,
        }
    )

datasets_summary_df = pd.DataFrame(dataset_rows).set_index("Dataset")
ma.print_caption("Table 0a", "Summary of training, validation, and held-out test datasets")
display(datasets_summary_df)
ma.print_note(
    "Shorthand labels in the first column are used throughout all tables and figures. "
    "Training datasets (Electricity, Drones) are used purely as respondent-level choice microdata via cross-entropy loss. "
    "OLS slopes = number of population regression slopes used for evaluation (Consumer rights spans two product conjoints of 19 slopes each = 38 total; "
    "AI & offshoring spans four country × arm regressions of 8 slopes each = 32 total). "
    "Flights (Goldfarb et al., 2026) is the validation set used to select each run's checkpoint (best.pt)."
)


Table 0a
Summary of training, validation, and held-out test datasets
Note. Shorthand labels in the first column are used throughout all tables and figures. Training datasets (Electricity, Drones) are used purely as respondent-level choice microdata via cross-entropy loss. OLS slopes = number of population regression slopes used for evaluation (Consumer rights spans two product conjoints of 19 slopes each = 38 total; AI & offshoring spans four country × arm regressions of 8 slopes each = 32 total). Flights (Goldfarb et al., 2026) is the validation set used to select each run's checkpoint (best.pt).


,Citation,Journal,Role,Choice task & key attributes,Respondents,Choices,OLS slopes
Dataset,,,,,,,
Electricity,Wu (2025),Energy Research & Social Science,Training (microdata),"Electricity-plan conjoint: monthly bill, renewable share, local renewables","1,861","18,591",—
Drones,Berke et al. (2023),Transportation Research Part C,Training (microdata),"Drone-delivery discrete choice: delivery fee, wait time, privacy","3,625","36,297",—
Flights,Goldfarb et al. (2026),Journal of Cleaner Production,Validation (selects best.pt),"Flight conjoint: price, sustainable aviation fuel, airline, baggage, legroom, seat","1,000","5,000",12
Consumer rights,Kantorowicz-Reznichenko et al. (2026),Journal of Law and Empirical Analysis,Held-out test (price study),"Laptop & laundry-machine conjoints: price, brand, product specs, warranty, privacy, liability, conflict resolution","2,242","26,823",38
AI & offshoring,Magistro et al. (2025),American Journal of Political Science,Held-out test (price study),"Smartphone AI & offshoring conjoint (US & Canada): price, customer service jobs, factory jobs, data science jobs","5,525","21,005",32
Food labels,Grummon et al. (2025),JAMA Network Open,Held-out test (non-price study),Six-arm nutrition-label trial: front-of-package warning labels vs. control,"5,633","33,798",5
Vaccines,Kreps et al. (2020),JAMA Network Open,Held-out test (non-price study),"COVID-19 vaccine conjoint: efficacy, duration, side effects, FDA, origin, endorsements","1,971","9,855",11


In [3]:
ma.print_caption("Table 0b", "Canonical microdata run and test_best inventory")
display(ctx.status_grid)
ma.print_note(
    "Each cell reports training status, selected validation checkpoint step (best.pt), and whether held-out test_best evaluations are present."
)


Table 0b
Canonical microdata run and test_best inventory
Note. Each cell reports training status, selected validation checkpoint step (best.pt), and whether held-out test_best evaluations are present.


,KEV-0.8B,KEV-4B,KEV-9B
Regime,,,
Off-the-shelf,step 0 [test_best ✓],step 0 [test_best ✓],step 0 [test_best ✓]
Electricity,finished (@640/1395) [test_best ✓],finished (@416/1395) [test_best ✓],finished (@928/1395) [test_best ✓]
Drones,finished (@1152/2730) [test_best ✓],finished (@544/2730) [test_best ✓],finished (@256/2730) [test_best ✓]
Drones + Electricity,finished (@1600/4125) [test_best ✓],finished (@1472/4125) [test_best ✓],finished (@1600/4125) [test_best ✓]
